This notebook produces the evaluation tsv correlations between true and predicted signal values.

For the enformer evaluations, we take the middle 4 bins to cover the 512 bp of the summit center. This produces the best correlations compared to other number of bins. This can easily be tested by change the slicing of `preds_raw`.

To take the 8 middle bins, grab `preds_raw[:, 444:452]`


In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from pathlib import Path
import os

ENFORMER_INPUTS = Path("/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/")
ENFORMER_OUTPUTS = Path("/home/azstephe/liverRegression/regression_liver/data/enformer_outputs/")
TARGET_TRACK = 205

Double check this is the correct track - we are interested in liver.

In [6]:
targets_mouse = pd.read_csv('https://raw.githubusercontent.com/calico/basenji/master/manuscripts/cross2020/targets_mouse.txt', sep='\t')

# Accessing row 205 gives the info for global track 5518
print(targets_mouse.iloc[TARGET_TRACK])

index                                                       5518
genome                                                         1
identifier                                                 UW3.1
file           /home/drk/tillage/datasets/mouse/atac/uw-atlas...
clip                                                          32
scale                                                          4
sum_stat                                                    mean
description                ATAC:Hepatocytes-clusters_3-cluster_1
Name: 205, dtype: object


In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from pathlib import Path
import os

ENFORMER_INPUTS = Path("/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/")
ENFORMER_OUTPUTS = Path("/home/azstephe/liverRegression/regression_liver/data/enformer_outputs/")

results_list = []
meta_results = []
dir_list = ['log_test2', 'log_test1']

for subdir_name in dir_list:
# for subdir_name in os.listdir(ENFORMER_OUT):
    subdir_path = ENFORMER_OUTPUTS / subdir_name
    if not subdir_path.is_dir(): 
        continue

    # dict to collect averages for global metadata (per species)
    meta_collector = {}

    # change the keyword for your saved npy file
    for pred_path in subdir_path.glob("*_205_preds.npy"):
        species = pred_path.name.split('_')[0]

        # load enformer predictions and average the 4 middle bins (446:450)
        preds_raw = np.load(pred_path) # shape (N, 896)
        preds_avg = np.mean(preds_raw[:, 446:450], axis=1).flatten()

        # collect meta for all subdirs
        if species not in meta_collector:
            meta_collector[species] = []
        meta_collector[species].extend(preds_avg)

        # calculate correlations for positives only
        if subdir_name not in ['log_test1', 'neg']:
            bed_path = ENFORMER_INPUTS / subdir_name / f'{species}_liver_TEST_500bp.bed'
            
            if bed_path.exists():
                # load true values
                true_values = pd.read_csv(bed_path, header=None, sep=r'\s+').iloc[:, 4].values

                # check we are comparing the correct files - will error out
                if len(preds_avg) != len(true_values):
                    print(f"Warning: Size mismatch! Preds: {len(preds_avg)}, True: {len(true_values)}")

                r, p = pearsonr(true_values, preds_avg)
                s, sp = spearmanr(true_values, preds_avg)
                results_list.append({
                    'subdir': subdir_name, 'species': species,
                    'pearson_r': r, 'pearson_p': p, 
                    'spearman_rho': s, 'spearman_p': sp, 
                    'n': len(preds_avg)
                })

    # finalize metadata
    for species, all_preds in meta_collector.items():
        if all_preds:
            meta_results.append({
                'subdir': subdir_name,
                'species': species,
                'global_avg': np.mean(all_preds),
                'std_dev': np.std(all_preds),
                'count': len(all_preds)
            })

# save outputs
META_DIR = Path("/home/azstephe/liverRegression/regression_liver/data/enformer_meta/")
META_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame(results_list).to_csv(META_DIR / "correlations.tsv", sep='\t', index=False)
pd.DataFrame(meta_results).to_csv(META_DIR / "all_metadata.tsv", sep='\t', index=False)

# FOLDCHANGE SECTION

In [13]:
import pandas as pd
import scipy.stats
import matplotlib.pyplot as plt
import numpy as np

def pearson_spearman(x, y):
    pearson_corr, pearson_p_value = scipy.stats.pearsonr(x, y)
    print(f"Pearson correlation coefficient: {pearson_corr:.3f}, p-value: {pearson_p_value:.3g}")

    spearman_corr, spearman_p_value = scipy.stats.spearmanr(x, y)
    print(f"Spearman correlation coefficient: {spearman_corr:.3f}, p-value: {spearman_p_value:.3g}")
    
def collapse_if_identical(lst):
    return lst[0] if all(x == lst[0] for x in lst) else lst
    

# takes in list and returns peak with largest signal
def keep_largest_signal(lst, qn_df):
    if isinstance(lst, list):
        maxPeakName = ''
        maxPeakStrength = 0
        for peak in lst:
            matching_rows = qn_df[qn_df[3] == peak]
            if not matching_rows.empty:
                signal = matching_rows[4].iloc[0]
                if signal > maxPeakStrength:
                    maxPeakStrength = signal
                    maxPeakName = peak
        return maxPeakName
    else: 
        return lst

def get_biggest_overlap(lst, col, df):
    maxOverlap = 0
    maxPeakOverlap = ""
    for peak in lst:
        row = df[df[col] == peak].iloc[0] #iloc bc possible to return multiple rows (never will based on th
        overlap = max(0, min(row[2], row[12])-max(row[1], row[11]))
        if overlap > maxOverlap:
            maxOverlap = overlap
            maxPeakOverlap = peak
    return peak

def mean_squared_error(x, y):
    x = np.asarray(x)
    y = np.asarray(y)
    return np.mean((x - y) ** 2)

In [20]:
import pandas as pd
import numpy as np
import scipy.stats
from pathlib import Path

ENFORMER_INPUTS = Path("/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/")
ENFORMER_OUTPUTS = Path("/home/azstephe/liverRegression/regression_liver/data/enformer_outputs/")
ONE_TO_ONE_DIR = Path("/home/azstephe/liverRegression/regression_liver/data/test_splits/oneToOnePeaks/")

species_list = ['macaque', 'rat', 'cow', 'pig']
model_name = 'enformer_205' 

def get_signal_dict(bed_path, pred_path):
    """Maps Peak ID to the averaged Enformer prediction and True value."""
    # 1. Load True Values from BED (Col 3 is Name, Col 4 is Signal)
    bed = pd.read_csv(bed_path, sep=r'\s+', header=None)
    true_map = dict(zip(bed[3], bed[4]))
    
    # 2. Load Predictions (Assuming order matches BED)
    preds_raw = np.load(pred_path)
    preds_avg = np.mean(preds_raw[:, 446:450], axis=1).flatten()
    
    # 3. Create a ID -> Pred map based on order
    pred_map = dict(zip(bed[3], preds_avg))
    
    return true_map, pred_map

all_results = []

for species in species_list:    
    # 1. Load the "Bridge" (One-to-One Orthologs)
    # col 4: non-mouse ID, col 14: mouse ID
    bridge_path = ONE_TO_ONE_DIR / f"{species}_mouse.bed"
    if not bridge_path.exists(): continue
    bridge = pd.read_csv(bridge_path, header=None, sep='\t')
    
    # 2. Load Data for Mouse (Positive Test Set)
    m_true, m_pred = get_signal_dict(
        ENFORMER_INPUTS / "log_pos" / "mouse_liver_TEST_500bp.bed",
        ENFORMER_OUTPUTS / "log_pos" / "mouse_liver_TEST_enformer_205_preds.npy"
    )
    
    # 3. Load Data for Non-Mouse
    s_true, s_pred = get_signal_dict(
        ENFORMER_INPUTS / "log_test2" / f"{species}_liver_TEST_500bp.bed",
        ENFORMER_OUTPUTS / "log_test2" / f"{species}_liver_TEST_enformer_205_preds.npy"
    )

    # 4. Align using the Bridge
    data = []
    for _, row in bridge.iterrows():
        s_id, m_id = row[4], row[14]
        if m_id in m_true and s_id in s_true:
            data.append({
                'm_true': m_true[m_id], 'm_pred': m_pred[m_id],
                's_true': s_true[s_id], 's_pred': s_pred[s_id]
            })
    
    df = pd.DataFrame(data)
    
    # 5. Calculate Fold Changes (Delta Log)
    df['true_diff'] = df['m_true'] - df['s_true']
    df['pred_diff'] = df['m_pred'] - df['s_pred']
    
    # 6. Statistical Metrics
    p, pp = scipy.stats.pearsonr(df['true_diff'], df['pred_diff'])
    s, sp = scipy.stats.spearmanr(df['true_diff'], df['pred_diff'])
    
    # Same sign (Directional Accuracy)
    same_sign = (np.sign(df['true_diff']) == np.sign(df['pred_diff'])).mean()
    
    all_results.append({
        'Species': species,
        'Pearson_r': p,
        'Pearson_p': pp,
        'Spearman_rho': s,
        'Spearman_p': sp,
        'Directional_Accuracy': same_sign,
        'N': len(df)
    })

results_df = pd.DataFrame(all_results)

META_DIR = Path("/home/azstephe/liverRegression/regression_liver/data/enformer_meta/")
results_df.to_csv(META_DIR / "foldchange.tsv", sep='\t', index=False)

results_df

,Species,Pearson_r,Pearson_p,Spearman_rho,Spearman_p,Directional_Accuracy,N
0,macaque,0.248452,5.000716e-07,0.225294,5.501994e-06,0.588972,399
1,rat,0.281193,1.561601e-14,0.285924,5.398304e-15,0.591099,719
2,cow,0.331787,2.385769e-09,0.330908,2.643564e-09,0.561688,308
3,pig,0.166881,5.798422e-03,0.182050,2.579644e-03,0.569853,272


In [25]:
import pandas as pd
import numpy as np
import scipy.stats
from pathlib import Path
from sklearn.metrics import mean_squared_error

ENFORMER_INPUTS = Path("/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/")
ENFORMER_OUTPUTS = Path("/home/azstephe/liverRegression/regression_liver/data/enformer_outputs/")
ONE_TO_ONE_DIR = Path("/home/azstephe/liverRegression/regression_liver/data/test_splits/oneToOnePeaks/")
META_DIR = Path("/home/azstephe/liverRegression/regression_liver/data/enformer_meta/")
META_DIR.mkdir(parents=True, exist_ok=True)

species_list = ['macaque', 'rat', 'cow', 'pig']

def format_result_value(metric, value):
    """Format values based on the metric type provided."""
    if "Count" in metric:
        return int(value)
    elif "P-Val" in metric:
        return f"{value:.2E}"
    elif "%" in metric or metric in ["Pearson", "Spearman", "MSE"]:
        return round(float(value), 3)
    return value
    
def get_signal_dict(bed_path, pred_path):
    """Maps Peak ID to the averaged Enformer prediction and True value."""
    bed = pd.read_csv(bed_path, sep=r'\s+', header=None)
    true_map = dict(zip(bed[3], bed[4]))
    
    preds_raw = np.load(pred_path)
    # Average the middle 4 bins (512bp)
    preds_avg = np.mean(preds_raw[:, 446:450], axis=1).flatten()
    
    pred_map = dict(zip(bed[3], preds_avg))
    return true_map, pred_map

rows = []

for species in species_list:
    bridge_path = ONE_TO_ONE_DIR / f"{species}_mouse.bed"
    if not bridge_path.exists(): continue
    bridge = pd.read_csv(bridge_path, header=None, sep='\t')
    
    # Load Data (Mouse is in log_pos, Others in log_test2)
    m_true, m_pred = get_signal_dict(
        ENFORMER_INPUTS / "log_pos" / "mouse_liver_TEST_500bp.bed",
        ENFORMER_OUTPUTS / "log_pos" / "mouse_liver_TEST_enformer_205_preds.npy"
    )
    
    s_true, s_pred = get_signal_dict(
        ENFORMER_INPUTS / "log_test2" / f"{species}_liver_TEST_500bp.bed",
        ENFORMER_OUTPUTS / "log_test2" / f"{species}_liver_TEST_enformer_205_preds.npy"
    )

    # Align orthologs
    data = []
    for _, row in bridge.iterrows():
        s_id, m_id = row[4], row[14]
        if m_id in m_true and s_id in s_true:
            data.append({
                'm_true': m_true[m_id], 'm_pred': m_pred[m_id],
                's_true': s_true[s_id], 's_pred': s_pred[s_id]
            })
    
    df = pd.DataFrame(data)
    if df.empty: continue

    # Calculate Deltas (Fold Change in Log space)
    x = df['m_true'] - df['s_true']
    y = df['m_pred'] - df['s_pred']
    
    # Calculate Stats
    pearson, pp = scipy.stats.pearsonr(x, y)
    spearman, ps = scipy.stats.spearmanr(x, y)
    mse = mean_squared_error(x, y)
    
    num_ss = (np.sign(x) == np.sign(y)).sum()
    perc_ss = num_ss / len(df)

    # Append in the requested format
    metrics = [
        ('Same Sign Count', num_ss),
        ('Total Count', len(df)),
        ('Same Sign %', perc_ss),
        ('Pearson', pearson),
        ('Pearson P-Val', pp),
        ('Spearman', spearman),
        ('Spearman P-Val', ps),
        ('MSE', mse)
    ]

    for metric_name, value in metrics:
        formatted_val = format_result_value(metric_name, value)
        rows.append({
            'Species': species,
            'Group': 'Test2',
            'Metric': metric_name,
            'Enformer': formatted_val
        })

# Finalize DataFrame
results_df = pd.DataFrame(rows)

# Save to TSV
output_path = META_DIR / "foldchange_results.tsv"
results_df.to_csv(output_path, sep='\t', index=False)